# Train YOLOv3 on PCB defects (Colab, T4 GPU)

Trains the YOLOv3 transfer-learning model from this repo on a T4 **GPU**, then exports an **OpenVINO IR** (`yolov3.xml` + `yolov3.bin`) you can download.

**Before you start:**
1. `Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU**` (not TPU — this is a custom training loop that runs on GPU).
2. On your computer, zip the dataset and upload the single zip to Google Drive (`MyDrive`):
   ```bash
   cd ~/Downloads/PCB_yolov3/datasets
   zip -r -q ../unified_pku_yolo_gray640.zip unified_pku_yolo_gray640
   # then upload unified_pku_yolo_gray640.zip to Google Drive (MyDrive root)
   ```
   One zip uploads far faster than 26k loose files. Use the `*_gray640` set: it is already grayscale and letterboxed to 640×640 with matching labels.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi -L
import tensorflow as tf
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Install OpenVINO + deps (Colab already has GPU-enabled TensorFlow)

In [ ]:
!pip install -q "openvino>=2025.0" opencv-python-headless tensorboard
import openvino as ov; print('OpenVINO', ov.__version__)

## 3. Get the training code (this repo)

In [ ]:
import os
if not os.path.isdir('PCB_yolov3'):
    !git clone -q https://github.com/EasonLi292/PCB_yolov3.git
%cd PCB_yolov3
!ls scripts

## 4. Load your data from Google Drive
Mounts Drive, copies the zip to fast local Colab storage, and extracts it. Training then reads from `/content/data/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_NAME = 'unified_pku_yolo_gray640'          # change to dspcbsd_yolo_gray640 etc. if desired
ZIP_ON_DRIVE = f'/content/drive/MyDrive/{DATASET_NAME}.zip'

!cp "{ZIP_ON_DRIVE}" /content/
!mkdir -p /content/data && unzip -q -o /content/{DATASET_NAME}.zip -d /content/data
DATA_DIR = f'/content/data/{DATASET_NAME}'
print('DATA_DIR =', DATA_DIR)
# sanity check: counts per split + data.yaml present
import pathlib
for s in ['train', 'val', 'test']:
    d = pathlib.Path(DATA_DIR) / s / 'images'
    print(f'  {s}: {len(list(d.glob("*"))) if d.is_dir() else 0} images')
assert (pathlib.Path(DATA_DIR) / 'data.yaml').exists(), 'data.yaml not found — check the zip structure'

## 5. Download the pretrained Darknet weights (237 MB)

In [ ]:
!mkdir -p weights
![ -f weights/yolov3.weights ] || wget -q --show-progress https://pjreddie.com/media/files/yolov3.weights -O weights/yolov3.weights
!ls -lh weights/yolov3.weights

## 6. Train
Transfers the Darknet-53 backbone + FPN necks from the pretrained weights and trains the redefined detection heads (your classes). Backbone is frozen by default; add `--unfreeze` for a full fine-tune at a low `--lr` once the heads have warmed up.

On a T4, `--batch 8` at 640×640 is a safe fit. Drop to 4 if you hit OOM.

In [ ]:
EPOCHS = 40
BATCH = 8
!python scripts/train_yolov3.py --data "{DATA_DIR}" --weights weights/yolov3.weights --epochs {EPOCHS} --batch {BATCH}

## 7. Export to OpenVINO IR (.xml + .bin)

In [ ]:
RUN_DIR = f'runs/{DATASET_NAME}'
!python scripts/export_openvino.py --saved-model {RUN_DIR}/saved_model --out {RUN_DIR}/openvino
!ls -lh {RUN_DIR}/openvino

## 8. Bundle and download the results
Packages the IR (`yolov3.xml` + `yolov3.bin`) and `classes.txt` into one zip, downloads it, and also copies it to Drive as a backup. Drop this into `runs/<dataset>/openvino/` on your computer and run `scripts/analyze_openvino.py` locally.

In [ ]:
import shutil, os
src = f'{RUN_DIR}/openvino'
if os.path.exists(f'{RUN_DIR}/classes.txt'):
    shutil.copy(f'{RUN_DIR}/classes.txt', src)
out_zip = shutil.make_archive('/content/yolov3_pcb_ir', 'zip', src)
print('bundled:', out_zip)
!cp /content/yolov3_pcb_ir.zip /content/drive/MyDrive/   # backup to Drive
from google.colab import files
files.download('/content/yolov3_pcb_ir.zip')

### (optional) keep the Keras weights too
Lets you resume / fine-tune later without retraining from the backbone.

In [ ]:
!cp {RUN_DIR}/yolov3_best.weights.h5 /content/drive/MyDrive/{DATASET_NAME}_yolov3_best.weights.h5
print('saved best weights to Drive')